In [1]:
import pandas as pd
import numpy as np
import torch

In [2]:
url = "https://huggingface.co/datasets/yandex/yambda/resolve/main/sequential/50m/listens.parquet"
df = pd.read_parquet(url)
print(f"Загружено {len(df)} пользователей")
df.head()

Загружено 9238 пользователей


,uid,timestamp,item_id,is_organic,played_ratio_pct,track_length_seconds
0,100,"[39420, 39420, 39625, 40110, 40360, 40380, 406...","[8326270, 1441281, 286361, 732449, 3397170, 78...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[100, 100, 100, 100, 46, 100, 100, 100, 100, 1...","[170, 105, 185, 240, 130, 205, 205, 145, 95, 2..."
1,200,"[14329075, 14329075, 14329075, 14329075, 14329...","[3285270, 5253582, 9155883, 6113313, 1142869, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[9, 28, 42, 32, 0, 0, 5, 9, 100, 100, 100, 6, ...","[170, 170, 205, 155, 215, 185, 205, 195, 165, ..."
2,300,"[54090, 54100, 54110, 54120, 54125, 54135, 541...","[618910, 8793425, 8842650, 5806319, 8495320, 9...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[2, 4, 4, 5, 1, 5, 7, 1, 2, 2, 10, 6, 1, 23, 6...","[270, 130, 225, 210, 300, 210, 190, 340, 240, ..."
3,500,"[22695440, 22695690, 22695715, 22696095, 22696...","[6417502, 6896222, 6896938, 6950641, 7981221, ...","[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, ...","[100, 37, 10, 37, 100, 100, 11, 13, 20, 100, 1...","[225, 210, 240, 165, 150, 245, 135, 165, 275, ..."
4,600,"[1329190, 1329405, 1329690, 1329950, 1330185, ...","[8077497, 1865247, 95237, 4624864, 8332575, 44...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[100, 100, 100, 100, 100, 100, 100, 100, 100, ...","[245, 215, 235, 260, 250, 280, 180, 195, 155, ..."


In [3]:
def filter_listen_threshold(df, listen_threshold=50):
  # фильтруем пользователей по порогу в 50 процентов прослушиваний
  filtered_data = []
  for _, row in df.iterrows():
    items = np.array(row['item_id'])
    ratios = np.array(row['played_ratio_pct'])
    mask = ratios >= listen_threshold
    # если есть ХОТЬ ОДИН трек с ≥50% оставляем пользователя
    if mask.any():
      filtered_data.append({
      'uid': row['uid'],
      'item_id': items[mask].tolist(),
      'played_ratio_pct': ratios[mask].tolist()
          })
  df_filtered = pd.DataFrame(filtered_data)
  print(f"После фильтрации: {len(df_filtered)} пользователей")
  return df_filtered

In [4]:
df_filtered = filter_listen_threshold(df, listen_threshold=50)

После фильтрации: 9209 пользователей


In [5]:
df_filtered.head()

,uid,item_id,played_ratio_pct
0,100,"[8326270, 1441281, 286361, 732449, 7849270, 14...","[100, 100, 100, 100, 100, 100, 100, 100, 100, ..."
1,200,"[173016, 8420846, 8085657, 2999079, 299843, 96...","[100, 100, 100, 97, 81, 100, 100, 100, 100, 10..."
2,300,"[6961827, 1926141, 6442688, 6442688, 5196757, ...","[56, 76, 100, 100, 62, 58, 100, 66, 55, 76, 10..."
3,500,"[6417502, 7981221, 4119982, 8350060, 2943786, ...","[100, 100, 100, 100, 100, 100, 100, 100, 100, ..."
4,600,"[8077497, 1865247, 95237, 4624864, 8332575, 44...","[100, 100, 100, 100, 100, 100, 100, 100, 100, ..."


In [6]:
def get_track_index(df):
    # заменяем огромные ID треков из датасета на компактные индексы для уменьшение размера таблицы эмбеддингов модели
    all_items = set()
    for item in df['item_id']:
        all_items.update(item)
    track_to_index = {item: i+1 for i, item in enumerate(sorted(all_items))}
    print(f"Уникальных треков: {len(track_to_index)}")
    return track_to_index

In [7]:
track_to_index = get_track_index(df_filtered)

Уникальных треков: 631003


In [8]:
list(track_to_index.items())[:5]

[(26, 1), (43, 2), (50, 3), (71, 4), (105, 5)]

In [9]:
def split_users(df, validate_ratio=0.15, test_ratio=0.15):
    # разбиваем датасет на тренировочные данные валидационные и тестовые
    n_users = len(df)
    n_test = int(n_users * test_ratio)
    n_validate = int(n_users * validate_ratio)
    df_sorted = df.sort_values('uid').reset_index(drop=True)

    test_df = df_sorted.iloc[-n_test:].copy()
    validate_df = df_sorted.iloc[-n_test-n_validate:-n_test].copy()
    train_df = df_sorted.iloc[:-n_test-n_validate].copy()
    print(f"Train: {len(train_df)}, Validate: {len(validate_df)}, Test: {len(test_df)}")
    return train_df, validate_df, test_df

In [10]:
train_df, validate_df, test_df = split_users(df_filtered)

Train: 6447, Validate: 1381, Test: 1381


In [11]:
def encode_histories(df, track_to_index, max_seq_len=50):
    # обрезаем историю пользователя до последних 50 треков и заменяем ID треков на индексы из словаря
    result = []
    for _, row in df.iterrows():
        items = row['item_id'][-max_seq_len:]
        mapped_list = [track_to_index.get(item, 0) for item in items]
        result.append({
            'uid': row['uid'],
            'item_id': mapped_list,
            'played_ratio_pct': row['played_ratio_pct'][-max_seq_len:]
        })
    encode_df = pd.DataFrame(result)
    return encode_df

In [12]:
train_df = encode_histories(train_df, track_to_index)
validate_df = encode_histories(validate_df, track_to_index)
test_df = encode_histories(test_df, track_to_index)

In [13]:
train_df.head()

,uid,item_id,played_ratio_pct
0,100,"[341720, 288670, 378208, 499765, 204021, 31797...","[100, 100, 100, 100, 100, 100, 100, 100, 100, ..."
1,200,"[291986, 376458, 122009, 445846, 220767, 55082...","[100, 100, 100, 100, 100, 100, 101, 100, 100, ..."
2,300,"[343558, 127456, 543896, 488820, 494365, 24696...","[100, 100, 100, 100, 100, 100, 55, 100, 100, 1..."
3,500,"[431507, 536531, 277094, 561350, 198052, 54596...","[100, 100, 100, 100, 100, 100, 100, 100, 100, ..."
4,600,"[475016, 31604, 320336, 331161, 253287, 246698...","[100, 100, 100, 100, 100, 100, 100, 100, 100, ..."


In [14]:
validate_df.head()

,uid,item_id,played_ratio_pct
0,700300,"[176120, 2880, 10748, 137889, 622771, 49149, 3...","[100, 100, 100, 100, 100, 100, 100, 100, 100, ..."
1,700400,"[186757, 460285, 125471, 448290, 211501, 16813...","[100, 100, 100, 63, 100, 100, 100, 100, 100, 1..."
2,700500,"[583955, 264538, 466471, 510179, 25960, 534817...","[100, 100, 100, 100, 100, 100, 100, 100, 100, ..."
3,700600,"[463862, 254071, 454579, 46574, 147149, 206007...","[100, 100, 100, 71, 100, 100, 100, 100, 100, 1..."
4,700700,"[519662, 336969, 358715, 138381, 474812, 52083...","[100, 100, 100, 100, 100, 100, 100, 100, 100, ..."


In [15]:
test_df.head()

,uid,item_id,played_ratio_pct
0,848800,"[225971, 421265, 84626, 617209, 490720, 417388...","[100, 100, 100, 100, 100, 100, 100, 100, 100, ..."
1,848900,"[399717, 321282]","[78, 54]"
2,849200,"[139553, 564201, 175608, 582996, 248607, 53780...","[100, 100, 100, 100, 100, 84, 100, 100, 100, 1..."
3,849300,"[614529, 314630, 112339, 373722, 310773, 16476...","[100, 100, 100, 100, 100, 100, 100, 100, 100, ..."
4,849400,"[264699, 97496, 124471, 131648, 98092, 508614,...","[100, 100, 67, 100, 50, 100, 100, 100, 55, 67,..."


In [16]:
def create_tensors(df, cnt_item, max_seq_len=50):
  # превращаем таблицу в тензоры для модели
    inputs, targets, weights = [], [], []

    for _, row in df.iterrows():
        items = row['item_id']
        ratios = row.get('played_ratio_pct', [1.0] * len(items))

        for i in range(1, len(items)):
            history = items[max(0, i - max_seq_len):i]
            padded = [0] * (max_seq_len - len(history)) + history

            inputs.append(padded)
            targets.append(items[i])
            wgt = min(ratios[i] / 100.0, 1.0)
            weights.append(wgt)

    inputs_tensor = torch.tensor(inputs, dtype=torch.long)
    targets_tensor = torch.tensor(targets, dtype=torch.long)
    weights_tensor = torch.tensor(weights, dtype=torch.float32)

    return inputs_tensor, targets_tensor, weights_tensor

In [17]:
cnt_item = len(track_to_index)

# создаём тензоры
train_inputs, train_targets, train_weights = create_tensors(train_df, cnt_item)
validate_inputs, validate_targets, validate_weights = create_tensors(validate_df, cnt_item)
test_inputs, test_targets, test_weights = create_tensors(test_df, cnt_item)

# проверяем размерности
print(f"Train: inputs {train_inputs.shape}, targets {train_targets.shape}, weights {train_weights.shape}")
print(f"Validate:   inputs {validate_inputs.shape}, targets {validate_targets.shape}, weights {validate_weights.shape}")
print(f"Test:  inputs {test_inputs.shape}, targets {test_targets.shape}, weights {test_weights.shape}")

Train: inputs torch.Size([305682, 50]), targets torch.Size([305682]), weights torch.Size([305682])
Validate:   inputs torch.Size([65590, 50]), targets torch.Size([65590]), weights torch.Size([65590])
Test:  inputs torch.Size([65648, 50]), targets torch.Size([65648]), weights torch.Size([65648])


In [18]:
torch.save({
    # тензоры для обучения
    'train_inputs': train_inputs,
    'train_targets': train_targets,
    'train_weights': train_weights,

    # тензоры для валидации
    'validate_inputs': validate_inputs,
    'validate_targets': validate_targets,
    'validate_weights': validate_weights,

    # тензоры для тестирования
    'test_inputs': test_inputs,
    'test_targets': test_targets,
    'test_weights': test_weights,

    # доп. информация
    'cnt_item': cnt_item,
    'track_to_index': track_to_index
}, 'preprocessed_data.pt')

В процессе предобработки было выполнено:
- фильтрация данных по порогу прослушивания, что позволило исключить неосознанные взаимодействия и неактивных пользователей
- создан словарь уникальных треков с отображением оригинальных идентификаторов в индексы модели
- датасет разбит на обучающую, валидационную и тестовую выборки, для каждой сформированы последовательности фиксированной длины с паддингом
- подготовлены тензоры и сохранены для последующего обучения модели